# Model Comparison

Load Models

In [1]:
import joblib
import pickle
import tensorflow as tf
import os

# Load sklearn-based models
lr_model = joblib.load('../models/logistic_regression_model.pkl')
lr_vectorizer = joblib.load('../models/logistic_regression_vectorizer.pkl')

svm_model = joblib.load('../models/svm_model.pkl')
svm_vectorizer = joblib.load('../models/svm_vectorizer.pkl')

rf_model = joblib.load('../models/random_forest_model.pkl')
rf_vectorizer = joblib.load('../models/random_forest_vectorizer.pkl')

# Load TensorFlow model
tf_model = tf.keras.models.load_model('../models/tensorflow_model.h5')
with open('../models/tensorflow_tokenizer.pkl', 'rb') as f:
    tf_tokenizer = pickle.load(f)

print("All models loaded successfully!")
print(f"Logistic Regression: {lr_model}")
print(f"SVM: {svm_model}")
print(f"Random Forest: {rf_model}")
print(f"TensorFlow: {tf_model}")

All models loaded successfully!
Logistic Regression: LogisticRegression(max_iter=1000, random_state=42)
SVM: SVC(kernel='linear', random_state=42)
Random Forest: RandomForestClassifier(random_state=42)
TensorFlow: <Sequential name=sequential, built=True>


Load Data

In [3]:
from sklearn.model_selection import train_test_split
import pandas as pd
import string
from nltk.corpus import stopwords

data = pd.read_csv("../Data/spam_ham_dataset.csv")
data.head()

data.shape

# sns.countplot(x="label", data=data)
# plt.show()

ham_msg = data[data["label"] == "ham"]
spam_msg = data[data["label"] == "spam"]

# Downsample Ham emails to match the number of Spam emails
ham_msg_balanced = ham_msg.sample(n=len(spam_msg), random_state=42)

# Combine balanced data
balanced_data = pd.concat([ham_msg_balanced, spam_msg]).reset_index(drop=True)

# Visualize the balanced dataset
# sns.countplot(x="label", data=balanced_data)
# plt.title("Balanced Distribution of Spam and Ham Emails")
# plt.xticks(ticks=[0, 1], labels=["Ham (Not Spam)", "Spam"])
# plt.show()

balanced_data["text"] = balanced_data["text"].str.replace("Subject", "")
balanced_data.head()

punctuations_list = string.punctuation


def remove_punctuations(text):
    temp = str.maketrans("", "", punctuations_list)
    return text.translate(temp)


balanced_data["text"] = balanced_data["text"].apply(lambda x: remove_punctuations(x))
balanced_data.head()


def remove_stopwords(text):
    stop_words = stopwords.words("english")

    imp_words = []

    # Storing the important words
    for word in str(text).split():
        word = word.lower()

        if word not in stop_words:
            imp_words.append(word)

    output = " ".join(imp_words)

    return output


balanced_data["text"] = balanced_data["text"].apply(lambda text: remove_stopwords(text))
balanced_data.head()

train_X, test_X, train_Y, test_Y = train_test_split(
    balanced_data["text"], balanced_data["label"], test_size=0.2, random_state=42
)

Model Accuracy Scores

In [11]:
from sklearn.metrics import classification_report
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

def evaluate_model(model, vectorizer, test_X, test_Y):
    test_X_vectors = vectorizer.transform(test_X)
    predictions = model.predict(test_X_vectors)
    print(classification_report(test_Y, predictions))

# Evaluate Logistic Regression Model
print("Evaluating Logistic Regression Model:")
evaluate_model(lr_model, lr_vectorizer, test_X, test_Y)

# Evaluate SVM Model
print("\nEvaluating SVM Model:")
evaluate_model(svm_model, svm_vectorizer, test_X, test_Y)

# Evaluate Random Forest Model
print("\nEvaluating Random Forest Model:")
evaluate_model(rf_model, rf_vectorizer, test_X, test_Y)

# Evaluate TensorFlow Model (matching training script exactly)
print("\nEvaluating TensorFlow Model:")

# Create and fit tokenizer on training data (same as training script)
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_X)

train_sequences = tokenizer.texts_to_sequences(train_X)
test_sequences = tokenizer.texts_to_sequences(test_X)

max_len = 100  # Maximum sequence length
train_sequences = pad_sequences(
    train_sequences, maxlen=max_len, padding="post", truncating="post"
)
test_sequences = pad_sequences(
    test_sequences, maxlen=max_len, padding="post", truncating="post"
)

# Convert labels to numeric
train_Y_numeric = (train_Y == "spam").astype(int)
test_Y_numeric = (test_Y == "spam").astype(int)

# Evaluate the model
test_loss, test_accuracy = tf_model.evaluate(test_sequences, test_Y_numeric)
print("Test Loss :", test_loss)
print("Test Accuracy :", test_accuracy)
print(classification_report(test_Y, ["spam" if pred == 1 else "ham" for pred in (tf_model.predict(test_sequences) > 0.5).astype("int32").flatten()]))

Evaluating Logistic Regression Model:
              precision    recall  f1-score   support

         ham       1.00      0.93      0.97       315
        spam       0.93      1.00      0.96       285

    accuracy                           0.96       600
   macro avg       0.97      0.97      0.96       600
weighted avg       0.97      0.96      0.97       600


Evaluating SVM Model:
              precision    recall  f1-score   support

         ham       0.99      0.98      0.99       315
        spam       0.98      0.99      0.99       285

    accuracy                           0.99       600
   macro avg       0.99      0.99      0.99       600
weighted avg       0.99      0.99      0.99       600


Evaluating Random Forest Model:
              precision    recall  f1-score   support

         ham       0.99      0.96      0.97       315
        spam       0.95      0.99      0.97       285

    accuracy                           0.97       600
   macro avg       0.97      0.97 

## Model Comparison Analysis

### Performance Summary

Based on the evaluation results above, here's how the four models performed:

| Model | Accuracy | Precision (Ham) | Recall (Ham) | Precision (Spam) | Recall (Spam) | F1-Score (Avg) |
|-------|----------|-----------------|--------------|------------------|---------------|----------------|
| **Logistic Regression** | 96% | 1.00 | 0.93 | 0.93 | 1.00 | 0.96 |
| **SVM** | **99%** | 0.99 | 0.98 | 0.98 | 0.99 | 0.99 |
| **Random Forest** | 97% | 0.99 | 0.96 | 0.95 | 0.99 | 0.97 |
| **TensorFlow LSTM** | 96% | 0.97 | 0.96 | 0.96 | 0.97 | 0.96 |

### Key Findings

1. **Support Vector Machine (SVM)** - **Winner**
   - Achieved the highest accuracy at **99%**
   - Excellent balance with 0.99 precision and 0.99 recall for both classes
   - Best weighted F1-score of 0.99
   - Near-perfect performance with minimal false positives and false negatives

2. **Random Forest**
   - Strong second place with **97% accuracy**
   - High precision (0.99) for ham emails
   - Excellent recall (0.99) for spam detection
   - Good balance between performance and interpretability

3. **Logistic Regression & TensorFlow LSTM** (Tied)
   - Both achieved **96% accuracy**
   - Logistic Regression: Perfect precision on ham (1.00) but lower recall (0.93)
   - TensorFlow: More balanced but slightly lower overall performance than expected

### Recommendation: **Best Model for Spam Email Detection**

**Winner: Support Vector Machine (SVM)**

**Rationale:**
- **Highest Overall Accuracy**: 99% accuracy significantly outperforms other models
- **Best Balanced Performance**: 0.99 precision and 0.99 recall for both spam and ham classes
- **Minimal False Positives/Negatives**: Only 1% error rate means users will rarely see legitimate emails marked as spam or vice versa
- **Efficient & Scalable**: Lower computational cost than deep learning while delivering superior results
- **Fast Predictions**: Quick inference time suitable for real-time email filtering

**Why SVM Outperformed TensorFlow LSTM:**
- The dataset benefits from TF-IDF feature representation, which captures word importance effectively
- SVM with linear kernel excels in high-dimensional sparse spaces (typical of text data)
- The LSTM model, while powerful for sequential data, may be slightly overfitted or the dataset size may not fully leverage its deep learning capabilities
- Traditional ML with proper feature engineering (TF-IDF) is highly effective for this spam detection task

**Production Deployment Recommendation:**
The **SVM model** is the clear choice for a production spam email detection system due to:
- Superior accuracy (99%)
- Fast prediction times
- Lower memory footprint
- Easier deployment and maintenance
- Excellent balance between catching spam and avoiding false positives

### Conclusion

The **SVM model** is the best performer for this spam email detection application, achieving 99% accuracy with excellent precision and recall. While deep learning models like LSTM are powerful, this comparison demonstrates that traditional machine learning with proper feature engineering (TF-IDF vectorization) can deliver superior results for text classification tasks, especially with moderate-sized datasets.